# MAI-Thinking-1 Architecture

Implementation of the model architecture described in the [MAI-Thinking-1 technical report](https://microsoft.ai/wp-content/uploads/2026/06/main_20260602_2.pdf) (Microsoft AI, 2026).

**Key architectural choices:**
- Decoder-only Transformer with **no biases** anywhere
- **Tied** input/output embedding weights
- **Dual RMSNorm** (pre + post) wrapping each sub-layer before the residual (Gemma 3 style)
- **Periodic 5:1 local/global attention**: every 6th layer is global; all others are local
  - Local layers: sliding-window attention + **RoPE** (base freq 10,000; window = 512 in full model)
  - Global layers: full causal attention + **NoPE** (zero positional encoding)
- **Alternating Dense/MoE FFN**: even layers use Dense SwiGLU, odd layers use LatentMoE
- **LatentMoE**: tokens are compressed to a latent space before expert dispatch; routing uses the original representation
- **GQA**: multiple query heads share each KV head; per-head QK-Norm applied before positional encoding


## Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Core Components

### RMSNorm

Used 4 times per block (pre/post for attention + pre/post for FFN) and once as the final model norm.


In [2]:
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(emb_dim))

    def forward(self, x):
        variance = x.pow(2).mean(dim=-1, keepdim=True)
        x_norm = x * torch.rsqrt(variance + self.eps)
        return (self.scale * x_norm).to(dtype=x.dtype)

### RoPE — Rotary Position Encoding

**Only used in local attention layers** (base frequency 10,000).
Global attention layers use **NoPE**: no positional encoding of any kind.


In [3]:
def compute_rope(head_dim, theta_base=10_000, context_length=512):
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(context_length).unsqueeze(1)    # (context_length, 1)
    angles    = positions * inv_freq.unsqueeze(0)             # (context_length, head_dim/2)
    angles    = torch.cat([angles, angles], dim=-1)           # (context_length, head_dim)
    return torch.cos(angles), torch.sin(angles)


def apply_rope(x, cos, sin, start_pos=0):
    """x: (batch, heads, seq, head_dim)"""
    seq_len = x.shape[2]
    cos = cos[start_pos:start_pos + seq_len].unsqueeze(0).unsqueeze(0)
    sin = sin[start_pos:start_pos + seq_len].unsqueeze(0).unsqueeze(0)
    half  = x.shape[-1] // 2
    x_rot = torch.cat([-x[..., half:], x[..., :half]], dim=-1)
    return (x * cos + x_rot * sin).to(dtype=x.dtype)

### Attention: Local (Sliding Window) vs Global (NoPE)

A single class handles both modes via `is_global`:

| Mode | Positional Encoding | Attention Span |
|------|--------------------|-----------------|
| **Local** | RoPE (base 10k) | Sliding window of `window_size` tokens |
| **Global** | None (NoPE) | Full sequence (standard causal) |

**QK-Norm** (RMSNorm on Q and K per head) is applied **before** positional encoding in both modes.
**GQA**: `n_q_heads // n_kv_heads` query heads share each KV pair.


In [4]:
class MAIAttention(nn.Module):
    """
    Unified GQA attention module for MAI-Thinking-1.
      Local  (is_global=False): sliding-window causal attention + RoPE + QK-Norm
      Global (is_global=True):  full causal attention            + NoPE + QK-Norm
    No biases anywhere.
    """
    def __init__(self, emb_dim, n_q_heads, n_kv_heads, head_dim,
                 context_length, window_size, is_global, dtype):
        super().__init__()
        assert n_q_heads % n_kv_heads == 0, 'n_q_heads must be divisible by n_kv_heads'
        self.n_q_heads  = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim   = head_dim
        self.is_global  = is_global

        q_dim  = n_q_heads  * head_dim
        kv_dim = n_kv_heads * head_dim

        self.W_q = nn.Linear(emb_dim, q_dim,  bias=False, dtype=dtype)
        self.W_k = nn.Linear(emb_dim, kv_dim, bias=False, dtype=dtype)
        self.W_v = nn.Linear(emb_dim, kv_dim, bias=False, dtype=dtype)
        self.W_o = nn.Linear(q_dim,   emb_dim, bias=False, dtype=dtype)

        # QK-Norm: RMSNorm per head dimension, before positional encoding
        self.q_norm = RMSNorm(head_dim)
        self.k_norm = RMSNorm(head_dim)

        if not is_global:
            # Local layers: pre-compute RoPE tables and sliding-window causal mask
            cos, sin = compute_rope(head_dim, theta_base=10_000, context_length=context_length)
            self.register_buffer('cos', cos)
            self.register_buffer('sin', sin)
            mask = self._sliding_window_mask(context_length, window_size)
            self.register_buffer('attn_mask', mask)

    @staticmethod
    def _sliding_window_mask(seq_len, window_size):
        """Additive attention mask: 0 where allowed, -inf where blocked."""
        rows = torch.arange(seq_len).unsqueeze(1)
        cols = torch.arange(seq_len).unsqueeze(0)
        causal    = cols <= rows                        # no future tokens
        in_window = cols >= (rows - window_size + 1)   # within sliding window
        mask = torch.zeros(seq_len, seq_len)
        mask[~(causal & in_window)] = float('-inf')
        return mask

    def forward(self, x, start_pos=0):
        b, seq_len, _ = x.shape

        q = self.W_q(x).view(b, seq_len, self.n_q_heads,  self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(b, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(b, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        # QK-Norm before positional encoding
        q = self.q_norm(q)
        k = self.k_norm(k)

        if self.is_global:
            # Global: full causal attention, no positional encoding (NoPE)
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=True)
        else:
            # Local: apply RoPE then restrict to sliding window
            q = apply_rope(q, self.cos, self.sin, start_pos)
            k = apply_rope(k, self.cos, self.sin, start_pos)
            mask = self.attn_mask[:seq_len, :seq_len]
            out  = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, enable_gqa=True)

        out = out.transpose(1, 2).reshape(b, seq_len, self.n_q_heads * self.head_dim)
        return self.W_o(out)

### FFN Option A: Dense SwiGLU

Used on **even-indexed** layers. Hidden dim expands 2x relative to `emb_dim`.

`SwiGLU: down( silu(gate(x)) * up(x) )`


In [5]:
class DenseSwiGLU(nn.Module):
    """Dense feed-forward with SwiGLU activation. No bias."""
    def __init__(self, emb_dim, hidden_dim, dtype):
        super().__init__()
        self.gate = nn.Linear(emb_dim, hidden_dim, bias=False, dtype=dtype)
        self.up   = nn.Linear(emb_dim, hidden_dim, bias=False, dtype=dtype)
        self.down = nn.Linear(hidden_dim, emb_dim, bias=False, dtype=dtype)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

### FFN Option B: LatentMoE (Sparse Mixture of Experts)

Used on **odd-indexed** layers. Based on the LatentMoE design (NVIDIA, 2025).

Data flow:
```
x (emb_dim)
 |-- router(x) -----------------------> select top-k experts   [routing on original repr]
 |-- compress(x) --> latent (emb_dim/2)
        |-- Expert_i (SwiGLU in latent space)                   [compute in compressed space]
        |-- weighted sum by router scores
               |-- expand --> emb_dim
```

The separation of routing (full `emb_dim`) from computation (compressed `latent_dim`) is the key LatentMoE innovation.


In [6]:
class Expert(nn.Module):
    """Single MoE expert: SwiGLU FFN in the compressed latent space."""
    def __init__(self, latent_dim, hidden_dim, dtype):
        super().__init__()
        self.gate = nn.Linear(latent_dim, hidden_dim, bias=False, dtype=dtype)
        self.up   = nn.Linear(latent_dim, hidden_dim, bias=False, dtype=dtype)
        self.down = nn.Linear(hidden_dim, latent_dim, bias=False, dtype=dtype)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class LatentMoE(nn.Module):
    """
    LatentMoE: shared compression -> route -> expert compute -> shared expansion.

    Full model: 512 experts, top-8, compression factor 2x, expert expansion 3x.
    Routing uses the original emb_dim representation (before compression).
    """
    def __init__(self, emb_dim, n_experts, top_k, latent_dim, expert_hidden_dim, dtype):
        super().__init__()
        self.top_k     = top_k
        self.n_experts = n_experts

        self.router   = nn.Linear(emb_dim,    n_experts,  bias=False, dtype=dtype)  # routes on original x
        self.compress = nn.Linear(emb_dim,    latent_dim, bias=False, dtype=dtype)  # shared down-proj
        self.experts  = nn.ModuleList([
            Expert(latent_dim, expert_hidden_dim, dtype) for _ in range(n_experts)
        ])
        self.expand   = nn.Linear(latent_dim, emb_dim,    bias=False, dtype=dtype)  # shared up-proj

    def forward(self, x):
        b, seq_len, emb_dim = x.shape
        x_flat = x.view(-1, emb_dim)                                      # (b*seq, emb_dim)

        # 1. Route on original (uncompressed) representation
        scores           = F.softmax(self.router(x_flat), dim=-1)         # (b*seq, n_experts)
        topk_w, topk_idx = scores.topk(self.top_k, dim=-1)                # (b*seq, top_k)
        topk_w           = topk_w / topk_w.sum(dim=-1, keepdim=True)      # normalize weights

        # 2. Shared compression before dispatch
        latent = self.compress(x_flat)                                     # (b*seq, latent_dim)

        # 3. Dispatch tokens to their selected experts and accumulate
        output = torch.zeros_like(latent)
        for k in range(self.top_k):
            idx = topk_idx[:, k]              # (b*seq,) — which expert each token uses
            w   = topk_w[:, k].unsqueeze(-1)  # (b*seq, 1) — router weight for this slot
            for e_id, expert in enumerate(self.experts):
                mask = (idx == e_id)
                if mask.any():
                    output[mask] += w[mask] * expert(latent[mask])

        # 4. Shared expansion back to emb_dim
        return self.expand(output).view(b, seq_len, emb_dim)

### MAI Transformer Block

Two **independent** periodic patterns co-exist in the same layer stack:

| Dimension | Period | Rule | Type |
|-----------|--------|------|------|
| Attention | 6 | `layer_idx % 6 == 5` | Global (NoPE, full causal) |
| Attention | 6 | otherwise | Local (RoPE, sliding window) |
| FFN | 2 | `layer_idx % 2 == 0` | Dense SwiGLU |
| FFN | 2 | `layer_idx % 2 == 1` | Sparse LatentMoE |

**Dual RMSNorm** (Gemma 3 style):
```python
x = x + post_norm(sublayer(pre_norm(x)))
```


In [7]:
class MAIBlock(nn.Module):
    """
    One MAI-Thinking-1 transformer layer.

    Attention and FFN periodicities run independently:
      Attention period 6: positions 0-4 are local, position 5 is global (then repeats).
      FFN       period 2: even positions are Dense, odd positions are LatentMoE.

    Dual RMSNorm wraps each sub-layer (Gemma 3 style):
      x = x + post_norm(sublayer(pre_norm(x)))
    """
    def __init__(self, layer_idx, cfg):
        super().__init__()
        self.layer_idx = layer_idx
        self.is_global = (layer_idx % 6 == 5)
        self.use_moe   = (layer_idx % 2 == 1)

        self.attn = MAIAttention(
            emb_dim        = cfg['emb_dim'],
            n_q_heads      = cfg['n_q_heads'],
            n_kv_heads     = cfg['n_kv_heads'],
            head_dim       = cfg['head_dim'],
            context_length = cfg['context_length'],
            window_size    = cfg['window_size'],
            is_global      = self.is_global,
            dtype          = cfg['dtype'],
        )

        self.ffn = LatentMoE(
            emb_dim           = cfg['emb_dim'],
            n_experts         = cfg['n_experts'],
            top_k             = cfg['top_k'],
            latent_dim        = cfg['latent_dim'],
            expert_hidden_dim = cfg['expert_hidden_dim'],
            dtype             = cfg['dtype'],
        ) if self.use_moe else DenseSwiGLU(
            emb_dim    = cfg['emb_dim'],
            hidden_dim = cfg['ffn_hidden_dim'],
            dtype      = cfg['dtype'],
        )

        # Four RMSNorms per block: pre/post for attention, pre/post for FFN
        self.pre_attn_norm  = RMSNorm(cfg['emb_dim'])
        self.post_attn_norm = RMSNorm(cfg['emb_dim'])
        self.pre_ffn_norm   = RMSNorm(cfg['emb_dim'])
        self.post_ffn_norm  = RMSNorm(cfg['emb_dim'])

    def forward(self, x, start_pos=0):
        # Attention sub-layer with dual norm
        x = x + self.post_attn_norm(self.attn(self.pre_attn_norm(x), start_pos))
        # FFN sub-layer with dual norm
        x = x + self.post_ffn_norm(self.ffn(self.pre_ffn_norm(x)))
        return x

### Full MAI Model

Notice: **no learned positional embedding** at the input. Position is handled inside each attention layer (RoPE for local, nothing for global).

In [8]:
class MAIModel(nn.Module):
    """
    MAI-Thinking-1 full architecture.

    Notable design choices from the paper:
      - No learned positional embedding (position handled per-layer: RoPE or NoPE)
      - No biases in any linear layer
      - Tied input and output embedding weights
      - n_layers must be a multiple of 6 (enforced by 5:1 attention ratio)
    """
    def __init__(self, cfg):
        super().__init__()
        assert cfg['n_layers'] % 6 == 0,                          'n_layers must be a multiple of 6'
        assert cfg['n_q_heads'] % cfg['n_kv_heads'] == 0,         'n_q_heads must be divisible by n_kv_heads'
        assert cfg['emb_dim'] == cfg['n_q_heads'] * cfg['head_dim'], 'emb_dim must equal n_q_heads * head_dim'

        self.tok_emb  = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'])
        self.blocks   = nn.ModuleList([MAIBlock(i, cfg) for i in range(cfg['n_layers'])])
        self.norm     = RMSNorm(cfg['emb_dim'])
        self.out_head = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False)

        # Tie output projection weights to token embedding (halves embedding parameter count)
        self.out_head.weight = self.tok_emb.weight

    def forward(self, x, start_pos=0):
        x = self.tok_emb(x)
        for block in self.blocks:
            x = block(x, start_pos)
        x = self.norm(x)
        return self.out_head(x)

## Toy Configuration & Visualisation

Scaled-down config for local exploration.

| Dimension | Toy config | Full MAI-Thinking-1 |
|-----------|-----------|---------------------|
| n_layers | 12 | 78 |
| emb_dim | 384 | 6656 |
| n_q_heads | 6 | 80 |
| n_kv_heads | 2 | 8 |
| head_dim | 64 | 128 |
| window_size | 64 | 512 |
| n_experts | 4 | 512 |
| top_k | 2 | 8 |
| Active params | ~7M | 35B |


In [9]:
TOY_CONFIG = {
    'vocab_size'        : 50_257,   # GPT-2 tokenizer vocabulary
    'context_length'    : 512,
    'n_layers'          : 12,       # must be multiple of 6  ->  2 x [L L L L L G]

    # Attention
    'n_q_heads'         : 6,        # query heads per layer
    'n_kv_heads'        : 2,        # key/value heads  (GQA: 3 query heads share each KV)
    'head_dim'          : 64,       # emb_dim / n_q_heads = 384 / 6
    'window_size'       : 64,       # local attention sliding window (full model: 512)

    # Model size
    'emb_dim'           : 384,      # n_q_heads x head_dim

    # Dense FFN (even layers)
    'ffn_hidden_dim'    : 768,      # 2x emb_dim

    # Sparse LatentMoE (odd layers)
    'n_experts'         : 4,        # full model: 512
    'top_k'             : 2,        # full model: 8
    'latent_dim'        : 192,      # emb_dim / 2  (shared compression before dispatch)
    'expert_hidden_dim' : 576,      # 3x latent_dim (expansion within each expert)

    'dtype'             : torch.float32,  # use bfloat16 on GPU
}

### Instantiate & print PyTorch model tree

In [10]:
model = MAIModel(TOY_CONFIG)
print(model)

MAIModel(
  (tok_emb): Embedding(50257, 384)
  (blocks): ModuleList(
    (0): MAIBlock(
      (attn): MAIAttention(
        (W_q): Linear(in_features=384, out_features=384, bias=False)
        (W_k): Linear(in_features=384, out_features=128, bias=False)
        (W_v): Linear(in_features=384, out_features=128, bias=False)
        (W_o): Linear(in_features=384, out_features=384, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ffn): DenseSwiGLU(
        (gate): Linear(in_features=384, out_features=768, bias=False)
        (up): Linear(in_features=384, out_features=768, bias=False)
        (down): Linear(in_features=768, out_features=384, bias=False)
      )
      (pre_attn_norm): RMSNorm()
      (post_attn_norm): RMSNorm()
      (pre_ffn_norm): RMSNorm()
      (post_ffn_norm): RMSNorm()
    )
    (1): MAIBlock(
      (attn): MAIAttention(
        (W_q): Linear(in_features=384, out_features=384, bias=False)
        (W_k): Linear(in_features=384, out_featu

### Layer-by-layer architecture table

Shows both periodic patterns running simultaneously across the 12-layer stack.

In [11]:
print(f"{'Layer':>5}  {'Attention Type':<30}  {'FFN Type':<30}  Pos Enc")
print("-" * 80)
for block in model.blocks:
    attn_str = ("Global  full causal"
                if block.is_global
                else f"Local   window={TOY_CONFIG['window_size']}")
    ffn_str  = (f"LatentMoE  {TOY_CONFIG['n_experts']} experts  top-{TOY_CONFIG['top_k']}"
                if block.use_moe
                else "Dense SwiGLU")
    pos_str  = "NoPE" if block.is_global else "RoPE"
    marker   = "  <-- end of group" if block.is_global else ""
    print(f"  {block.layer_idx:>3}  {attn_str:<30}  {ffn_str:<30}  {pos_str}{marker}")

Layer  Attention Type                  FFN Type                        Pos Enc
--------------------------------------------------------------------------------
    0  Local   window=64               Dense SwiGLU                    RoPE
    1  Local   window=64               LatentMoE  4 experts  top-2     RoPE
    2  Local   window=64               Dense SwiGLU                    RoPE
    3  Local   window=64               LatentMoE  4 experts  top-2     RoPE
    4  Local   window=64               Dense SwiGLU                    RoPE
    5  Global  full causal             LatentMoE  4 experts  top-2     NoPE  <-- end of group
    6  Local   window=64               Dense SwiGLU                    RoPE
    7  Local   window=64               LatentMoE  4 experts  top-2     RoPE
    8  Local   window=64               Dense SwiGLU                    RoPE
    9  Local   window=64               LatentMoE  4 experts  top-2     RoPE
   10  Local   window=64               Dense SwiGLU           

### Parameter counts per component

In [12]:
def n_params(m):
    return sum(p.numel() for p in m.parameters())

print(f"{'Component':<45}  {'Params':>12}")
print("-" * 60)
print(f"{'Token Embedding  (= Output head, tied weights)':<45}  {n_params(model.tok_emb):>12,}")
print(f"{'Final RMSNorm':<45}  {n_params(model.norm):>12,}")
print()

dense_total, moe_total = 0, 0
for block in model.blocks:
    attn_kind = "Global" if block.is_global else "Local "
    ffn_kind  = "MoE  " if block.use_moe   else "Dense"
    total     = n_params(block)
    attn_p    = n_params(block.attn)
    ffn_p     = n_params(block.ffn)
    label     = f"Layer {block.layer_idx:>2}  [{attn_kind} attn + {ffn_kind} FFN]"
    print(f"  {label:<43}  {total:>12,}   attn: {attn_p:,}  ffn: {ffn_p:,}")
    if block.use_moe:
        moe_total   += total
    else:
        dense_total += total

print()
print(f"{'Dense layers subtotal':<45}  {dense_total:>12,}")
print(f"{'MoE   layers subtotal':<45}  {moe_total:>12,}")
print()
print(f"{'TOTAL unique parameters':<45}  {n_params(model):>12,}")
print(f"  (out_head shares {n_params(model.tok_emb):,} weights with tok_emb, not double-counted)")

Component                                            Params
------------------------------------------------------------
Token Embedding  (= Output head, tied weights)    19,298,688
Final RMSNorm                                           384

  Layer  0  [Local  attn + Dense FFN]             1,279,616   attn: 393,344  ffn: 884,736
  Layer  1  [Local  attn + MoE   FFN]             1,870,976   attn: 393,344  ffn: 1,476,096
  Layer  2  [Local  attn + Dense FFN]             1,279,616   attn: 393,344  ffn: 884,736
  Layer  3  [Local  attn + MoE   FFN]             1,870,976   attn: 393,344  ffn: 1,476,096
  Layer  4  [Local  attn + Dense FFN]             1,279,616   attn: 393,344  ffn: 884,736
  Layer  5  [Global attn + MoE   FFN]             1,870,976   attn: 393,344  ffn: 1,476,096
  Layer  6  [Local  attn + Dense FFN]             1,279,616   attn: 393,344  ffn: 884,736
  Layer  7  [Local  attn + MoE   FFN]             1,870,976   attn: 393,344  ffn: 1,476,096
  Layer  8  [Local  attn + De

### Smoke test — forward pass

Verify the model runs end-to-end with a random batch of token ids.

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

model = model.to(device)
model.eval()

batch_size, seq_len = 2, 64
dummy_input = torch.randint(0, TOY_CONFIG['vocab_size'], (batch_size, seq_len)).to(device)

with torch.no_grad():
    logits = model(dummy_input)

print(f"Input  shape: {dummy_input.shape}")
print(f"Output shape: {logits.shape}  (batch, seq_len, vocab_size)")
print(f"Output shape matches expected: {logits.shape == (batch_size, seq_len, TOY_CONFIG['vocab_size'])}")

Device: cpu
Input  shape: torch.Size([2, 64])
Output shape: torch.Size([2, 64, 50257])  (batch, seq_len, vocab_size)
Output shape matches expected: True
